# SCREEN: метаданные homologous ELS cCRE

E2E: homologous pairs → ELS человека и мыши → SCREEN → Parquet.
Загрузка, фильтрация, batching, parsing и checkpointing реализованы в `eclare.data.screen`.
Используйте TSV и BED одной версии Registry; accession — `EH38E…` / `EM10E…`.
Зависимости устанавливаются из `requirements.txt` в корне проекта.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "eclare").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from eclare.data.screen import fetch_human_mouse_metadata

DATA_DIR = project_root / "data/encode"
FIELDS = [
    "biosample",
    "tissue",
    "sample_type",
    "life_stage",
    "assay",
    "zscore",
]


In [ ]:
import os
from getpass import getpass

# Ключ остаётся в окружении kernel, а не в сохранённом notebook.
if not os.environ.get("SCREEN_API_KEY"):
    os.environ["SCREEN_API_KEY"] = getpass("SCREEN API key: ")


## Загрузка

Функция проверяет первые три ID каждого вида перед полной загрузкой.
При прерывании повторите вызов: скачанные батчи переиспользуются из `.screen_cache`.
`ccre_id` добавляется автоматически; `tissue` соответствует SCREEN `ontology`.
Одна строка метаданных — измерение cCRE в experiment/file/assay; `NaN` может быть настоящим отсутствующим значением.


In [ ]:
pairs, human_meta, mouse_meta = fetch_human_mouse_metadata(
    DATA_DIR / "hg38-mm10-Homologous.tsv",
    DATA_DIR / "GRCh38-cCREs.ELS.bed",
    DATA_DIR / "mm10-cCREs.ELS.bed",
    fields=FIELDS,
    batch_size=200,
)


In [ ]:
print(f"ELS pairs: {len(pairs):,}")
print(f"Human metadata rows: {len(human_meta):,}")
print(f"Mouse metadata rows: {len(mouse_meta):,}")

display(pairs.head())
display(human_meta.head())
display(mouse_meta.head())


## Сохранение

Файлы сохраняются в рабочую папку notebook. Parquet через pandas/pyarrow сохраняет
`DataFrame.attrs` с покрытием и ключами измерений для последующего `enrich_ccre_metadata`.


In [ ]:
pairs.to_parquet("orthologous_els.parquet", index=False)
human_meta.to_parquet("human_metadata.parquet", index=False)
mouse_meta.to_parquet("mouse_metadata.parquet", index=False)
